![NVIDIA Logo](images/nvidia.png)

# Building Graphs Inside Stages

In this notebook we'll look at how to build computational graphs inside of custom stages, another prerequisite skill for being able to build non-linear pipelines sending and/or receiving messages on multiple input or output ports.

---

## Objectives

By the time you complete this notebook you will be able to:

- Create arbitrarily-sized computational graphs inside of custom stages.

---

## Imports

In [1]:
import logging
import typing

from IPython.display import Image

from morpheus.config import Config

from morpheus.pipeline import Pipeline
from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.pipeline.pass_thru_type_mixin import PassThruTypeMixin
from morpheus.pipeline.single_port_stage import SinglePortStage

from morpheus.pipeline.stage import Stage

from morpheus.stages.general.monitor_stage import MonitorStage
from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.output.in_memory_sink_stage import InMemorySinkStage

from morpheus.utils.logger import configure_logging, reset_logging

import mrc
import mrc.core.operators as ops

---

## Anatomy of a Stage Revisited Again

Returning once again to our helpful image about the anatomy of Morpheus stages, we see that inside of stages there lies a computational graph consisting, as is the case with all computational graphs, of nodes and edges.

![anatomy of a stage](images/anatomy-of-a-stage.png)

When we first looked at building custom stages we touched upon the fact that Morpheus pipelines can be thought of as pipeline graphs consisting of stage graphs. Every time we have created a custom stage we have, in its `_build_single` method, created a new node inside the stage, attached it with an edge to the stage's input node, which we now understand more clearly to be the stage's input port(s), and then returned the newly created node.

Here once again is our most basic passthrough node, this time with a few additional comments highlighting what we just stated.

In [2]:
class PassThruStage(PassThruTypeMixin, GpuAndCpuMixin, SinglePortStage):
    """A Simple Pass Through Stage"""

    @property
    def name(self) -> str:
        return "pass-thru"

    def accepted_types(self) -> tuple:
        return (typing.Any, )

    def supports_cpp_node(self) -> bool:
        return False

    def on_data(self, message: typing.Any) -> typing.Any:
        return message

    def _build_single(self, builder: mrc.Builder, input_node: mrc.SegmentObject) -> mrc.SegmentObject:
        # Here we create a new node inside this custom stage.
        node = builder.make_node(self.unique_name, ops.map(self.on_data))
        
        # Here we make an edge between the input node (one of the stage's input ports, in this case
        # its only port.)
        builder.make_edge(input_node, node)

        # Here we return the newly created edge, which can be connected (via edges) to other input ports (nodes)
        # in other stages.
        return node

---

## Custom Stage With Multiple Inner Nodes

Here we create yet another passthrough stage, but this time inside of `_build_single` we break from our earlier pattern and create an additional inner node. This additional node has been defined to map all data passing through it to the stage's `on_data` method (in this case a passthrough function).

Thus the following stage might be called a double passthrough stage. Please see the comments in its defintion.

In [3]:
class MultiNodePassThruStage(GpuAndCpuMixin, PassThruTypeMixin, Stage):

    def __init__(self, c: Config):
        super().__init__(c)

        self._create_ports(1, 1) # One input port, one output port

    @property
    def name(self) -> str:
        return "non-linear-pass-thru"

    def supports_cpp_node(self):
        return False
    
    def on_data(self, message: typing.Any) -> typing.Any:
        return message

    # Remember we use `_build` instead of `_build_single` when inheriting from `Stage`.
    def _build(self, builder: mrc.Builder, input_nodes: list[mrc.SegmentObject]) -> list[mrc.SegmentObject]:

        # In the typical way we create a new node that applies all data to the `on_data` callback.
        inner_node = builder.make_node("inner_node", ops.map(self.on_data))
        # In the typical way we connect the input node to this stage node via edge.
        builder.make_edge(input_nodes[0], inner_node)

        # But here we create a second node, also a passthrough.
        outgoing = builder.make_node("outgoing", ops.map(self.on_data))
        # This node we connect to the output port via edge.
        builder.make_edge(inner_node, outgoing)

        # We now return the 2nd of the nodes we created as the output port.
        return [outgoing]

---

## Build and Run the Double Passthrough Pipeline

Although we've made some changes in our custom class definition compared to how we did things earlier, we have still created a passthrough stage with a single input port and single output port. Here we build a pipeline to use it and run the pipeline to show that everything still works as expected.

In [ ]:
input_file = 'data/simple_user_log.jsonlines'

In [ ]:
config = Config()

In [ ]:
pipeline = Pipeline(config)

source = pipeline.add_stage(FileSourceStage(config, filename=input_file, iterative=False, repeat=100))
broadcast = pipeline.add_stage(MultiNodePassThruStage(config))
in_mem_sink = pipeline.add_stage(InMemorySinkStage(config))
monitor = pipeline.add_stage(MonitorStage(config, description="Pipeline throughput"))

pipeline.add_edge(source, broadcast)
pipeline.add_edge(broadcast, in_mem_sink)
pipeline.add_edge(in_mem_sink, monitor)

In [ ]:
pipeline.build()

In [ ]:
viz_file = './pipeline_visualizations/multi_passthrough.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
await pipeline.run_async()

In [ ]:
messages = in_mem_sink.get_messages()
messages[0].get_data()